In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
import time
from bs4 import BeautifulSoup
import pandas as pd
import requests

# --- Configuration Setup ---
City = "Karachi"
MAX_WORKERS = 5  # Safe parallel threads to avoid IP blocks

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(headers)


def extract_detail_data(item):
    """Fetches and parses a single detail page concurrently."""
    link = item["link"]
    property_name = item["property_name"]

    try:
        page = session.get(link, timeout=10)
        dpageSoup = BeautifulSoup(page.content, "html.parser")

        # Location (from detail page)
        try:
            location = dpageSoup.select_one('span[aria-label="Location"]').text.strip()
        except:
            location = ""

        # Price
        try:
            price = dpageSoup.select_one("span._63ea997b").text.strip()
        except:
            price = ""

        # Area
        try:
            area = dpageSoup.select_one('span[aria-label="Area"]').text.strip()
        except:
            area = ""

        # Bedrooms (Raw number e.g., "3")
        try:
            bedRoom = (
                dpageSoup.select_one('span[aria-label="Beds"]')
                .text.strip()
                .split()[0]
            )
        except:
            bedRoom = ""

        # Bathrooms (Raw number e.g., "3")
        try:
            bathroom = (
                dpageSoup.select_one('span[aria-label="Baths"]')
                .text.strip()
                .split()[0]
            )
        except:
            bathroom = ""

        # KITCHENS: Extract count if digits exist, 1 if tag exists, 0 if missing
        try:
            k_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Kitchen")'
            )
            if k_tag:
                nums = re.findall(r"\d+", k_tag.text)
                kitchen = int(nums[0]) if nums else 1
            else:
                kitchen = 0
        except:
            kitchen = 0

        # FLOOR: Only extract if explicit number exists, else empty ""
        try:
            floorNum = ""
            for span in dpageSoup.select('span._9121cbf9'):
                text = span.text.strip()
                if text.lower().startswith("floor:") and not text.lower().startswith("floors in building"):
                    parts = text.split(":")
                    if len(parts) > 1 and parts[-1].strip():
                        floorNum = parts[-1].strip()
                    break
        except:
            floorNum = ""

        # TOTAL FLOORS IN BUILDING: Only extract if explicit number exists, else empty ""
        try:
            el_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Floors in Building")'
            )
            if el_tag and ":" in el_tag.text:
                parts = el_tag.text.split(":")
                floorsInBuilding = parts[-1].strip() if len(parts) > 1 else ""
            else:
                floorsInBuilding = ""
        except:
            floorsInBuilding = ""

        # ELEVATORS: Extract count if digits exist, 1 if tag exists, 0 if missing
        try:
            el_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Elevators")'
            )
            if el_tag:
                nums = re.findall(r"\d+", el_tag.text)
                elevator = int(nums[0]) if nums else 1
            else:
                elevator = 0
        except:
            elevator = 0

        # PARKING: Extract count if digits exist, 1 if tag exists, 0 if missing
        try:
            p_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Parking Spaces")'
            )
            if p_tag:
                nums = re.findall(r"\d+", p_tag.text)
                parking = int(nums[0]) if nums else 1
            else:
                parking = 0
        except:
            parking = 0

        # SERVANT QUARTERS: Extract count if digits exist, 1 if tag exists, 0 if missing
        try:
            sq_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Servant Quarters")'
            )
            if sq_tag:
                nums = re.findall(r"\d+", sq_tag.text)
                servant_quarters = int(nums[0]) if nums else 1
            else:
                servant_quarters = 0
        except:
            servant_quarters = 0

        # Binary Features
        furnished = (
            1
            if dpageSoup.select_one('span._9121cbf9:-soup-contains("Furnished")')
            else 0
        )
        electricityBackup = (
            1
            if dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Electricity Backup")'
            )
            else 0
        )

        # BUILT YEAR: Only extract if a 4-digit year exists, else empty ""
        try:
            el_tag = dpageSoup.select_one(
                'span._9121cbf9:-soup-contains("Built in year")'
            )
            if el_tag:
                years = re.findall(r"\d{4}", el_tag.text)
                builtYear = years[0] if years else ""
            else:
                builtYear = ""
        except:
            builtYear = ""

        return {
            "property_name": property_name,
            "link": link,
            "address": item["location"],  # Renamed to address
            "location": location,         # New clean detail location
            "price": price,
            "area": area,
            "bedrooms": bedRoom,
            "bathrooms": bathroom,
            "kitchen": kitchen,
            "floor": floorNum,
            "total_floors": floorsInBuilding,
            "parking": parking,
            "elevator": elevator,
            "servant_quarters": servant_quarters,
            "electricity_backup": electricityBackup,
            "furnished": furnished,
            "built_year": builtYear,
        }

    except Exception:
        return None


# --- Execution Loop ---
start_page = int(input("Enter page number to start from: "))
total_pages = int(input("How many pages to scrape: "))

end_page = start_page + total_pages
houses = []

try:
    for pageNumber in range(start_page, end_page):
        # NEW URL PATTERN: Houses_Property
        url = f"https://www.zameen.com/Houses_Property/Karachi-2-{pageNumber}.html"

        # Retry logic (up to 3 attempts) to handle transient drops/blocks
        pageSoup = None
        listings = None
        for attempt in range(3):
            page = session.get(url, timeout=10)
            if page.status_code == 200:
                pageSoup = BeautifulSoup(page.content, "html.parser")
                listings = pageSoup.select_one("ul.e20beb46")
                if listings:
                    break
            time.sleep(2)

        if listings is None:
            print(f"No listings found on page {pageNumber}. Skipping...")
            continue

        items = []
        for soup in listings.select("li.a37d52f0"):
            try:
                property_name = soup.select_one("h2._36dfb99f").text.strip()
                link = soup.select_one("a.d870ae17")["href"]

                if link.startswith("/"):
                    link = "https://www.zameen.com" + link

                location = soup.select_one("div.db1aca2f").text.strip()

                items.append({
                    "property_name": property_name,
                    "link": link,
                    "location": location,
                })
            except:
                continue

        # Multithreaded concurrent extraction
        page_results = []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [
                executor.submit(extract_detail_data, item) for item in items
            ]

            for future in as_completed(futures):
                result = future.result()
                if result:
                    page_results.append(result)

        houses.extend(page_results)

        print(
            f"Page {pageNumber} completed | Extracted {len(page_results)} items"
            f" | Cumulative Total: {len(houses)}"
        )
        time.sleep(1.5)

except Exception as e:
    print("Scraping stopped/interrupted:", e)

finally:
    output_dir = "/content/drive/MyDrive/Data/Karachi/Houses"
    os.makedirs(output_dir, exist_ok=True)

    df = pd.DataFrame(houses)

    path = os.path.join(
        output_dir, f"houses_{City}_data_{start_page}_to_{end_page-1}.csv"
    )
    df.to_csv(path, index=False)

    print("\n----------------------------------------")
    print(f"Saved {len(df)} properties successfully!")
    print(f"Output File Path: {path}")

Enter page number to start from: 1
How many pages to scrape: 480
Page 1 completed | Extracted 25 items | Cumulative Total: 25
Page 2 completed | Extracted 25 items | Cumulative Total: 50
Page 3 completed | Extracted 25 items | Cumulative Total: 75
Page 4 completed | Extracted 25 items | Cumulative Total: 100
Page 5 completed | Extracted 25 items | Cumulative Total: 125
Page 6 completed | Extracted 25 items | Cumulative Total: 150
Page 7 completed | Extracted 25 items | Cumulative Total: 175
Page 8 completed | Extracted 25 items | Cumulative Total: 200
Page 9 completed | Extracted 25 items | Cumulative Total: 225
Page 10 completed | Extracted 25 items | Cumulative Total: 250
Page 11 completed | Extracted 25 items | Cumulative Total: 275
Page 12 completed | Extracted 25 items | Cumulative Total: 300
Page 13 completed | Extracted 25 items | Cumulative Total: 325
Page 14 completed | Extracted 25 items | Cumulative Total: 350
Page 15 completed | Extracted 25 items | Cumulative Total: 375
Pa